> **ARCHIVED** — Not used in final submission. Kept for reference.
> Requires v1 data paths. Do not rerun.


# Cluster-Based Permutation Test: Flow Effect on the MMN (Exp 2 / Passive)

Self-contained -- includes its own setup (Cells 1-2 below mirror your
`extract_measures` notebook's Cells 1-3, trimmed to only what this analysis
needs: no `evokeds_raw`, no window definitions, since neither applies to a
cluster-based test).

**Contrast tested:** flow modulation of the MMN, i.e.
`(Deviant - Standard)_high-flow - (Deviant - Standard)_low-flow`, across the
full post-stimulus epoch and all EEG channels -- no manually selected time
window or electrode subset. See `analysis_decisions_log.md` for full
methodological rationale (why TFCE, why two-tailed, why paired design).


## Setup: configuration and paths

In [1]:
from __future__ import annotations
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import time

def find_repo_root(marker="core") -> Path:
    """Walk up from the current directory until a folder containing
    `marker` (e.g. core/) is found -- avoids hardcoding folder depth,
    which breaks every time notebooks get moved."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")


sys.path.insert(0, str(find_repo_root()))
from core import config_flow, config_load
from core.utils import load_pkl, show_latest_pkls

# ==============================================================================
# Configuration — edit here, then run all
# ==============================================================================
PAPER:    str = "flow"     # this notebook targets the flow paper specifically
PARADIGM: str = "passive"  # Experiment 2

# ==============================================================================
# FIXED — do not change
# ==============================================================================
STUDY = "Active" if PARADIGM == "active" else "Passive"

if sys.platform == "win32":
    root = Path(f"L:/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives")
else:
    root = Path(f"/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives")

LOCK: str = "stim"
DATE_SOURCE = show_latest_pkls(root, PAPER, PARADIGM, "stim", "raw", None, None)
print(f"Paper: {PAPER.upper()} | Paradigm: {PARADIGM.upper()} | Lock: {LOCK.upper()}")


Latest matching files in raw/:
  [20260529] 20260529_passive_41p_cond-highflow_stim_raw.pkl
  [20260529] 20260529_passive_41p_cond-lowflow_stim_raw.pkl
  [unknown] 2026_05_22_passive_41p_cond-highflow_stim_raw.pkl

→ Most recent: 20260529
Paper: FLOW | Paradigm: PASSIVE | Lock: STIM


In [2]:
# ==============================================================================
# Derived labels and paths (auto — do not edit)
# ==============================================================================
assert LOCK == "stim", "This notebook is specifically for stimulus-locked analyses."

_cfg = config_flow if PAPER == "flow" else config_load
CONDITIONS: dict = _cfg.CONDITIONS
DEVIANT_LABEL = "target" if PARADIGM == "active" else "deviant"

input_path_diff = root / "grand_averages" / PAPER / "diff"

N_PARTICIPANTS = {
    "flow":  {"active": 36, "passive": 41},
    "load":  {"active": 48, "passive": 47},
}
n = N_PARTICIPANTS[PAPER][PARADIGM]

print(f"Condition labels: {list(CONDITIONS.keys())}")
print(f"Deviant label   : {DEVIANT_LABEL}")
print(f"Input diff path : {input_path_diff}")

Condition labels: ['High Flow', 'Low Flow']
Deviant label   : deviant
Input diff path : L:\EEG_Flow_data\Study_EEG_Flow_Passive\derivatives\grand_averages\flow\diff


## Load per-participant difference-wave evokeds (diff only -- no raw needed here)

In [3]:
def _build_fname_diff(cond_key: str) -> str:
    return f"{DATE_SOURCE}_{PARADIGM}_{n}p_cond-{cond_key}_stim_diff.pkl"


evokeds_diff: dict[str, dict] = {}
for plot_label, cond_key in CONDITIONS.items():
    evokeds_diff[plot_label] = load_pkl(input_path_diff / _build_fname_diff(cond_key))
    print(f"Loaded [{plot_label}] -- diff keys: {list(evokeds_diff[plot_label].keys())}")

Loaded [High Flow] -- diff keys: ['highflow_deviant-standard', 'highflow_novel-standard', 'highflow_deviant-novel']
Loaded [Low Flow] -- diff keys: ['lowflow_deviant-standard', 'lowflow_novel-standard', 'lowflow_deviant-novel']


## Analysis: cluster-based permutation test

In [4]:
import numpy as np
from mne.stats import spatio_temporal_cluster_1samp_test
from mne.channels import find_ch_adjacency

### Confirm High/Low flow labels and extract the MMN difference wave

`CONDITIONS` was already loaded in Setup above -- this just confirms the labels and pulls out the cached Deviant-minus-Standard evokeds for each.

In [5]:
print("Available condition labels:", list(CONDITIONS.keys()))

COND_HIGH_LABEL = "High Flow"
COND_LOW_LABEL  = "Low Flow"

cond_key_high = CONDITIONS[COND_HIGH_LABEL]
cond_key_low  = CONDITIONS[COND_LOW_LABEL]

# The MMN = Deviant-minus-Standard difference wave, already cached per participant
diff_event = f"{DEVIANT_LABEL}-standard"

evokeds_high = evokeds_diff[COND_HIGH_LABEL][f"{cond_key_high}_{diff_event}"]
evokeds_low  = evokeds_diff[COND_LOW_LABEL][f"{cond_key_low}_{diff_event}"]

print(f"High flow: {len(evokeds_high)} participants cached")
print(f"Low flow : {len(evokeds_low)} participants cached")

Available condition labels: ['High Flow', 'Low Flow']
High flow: 41 participants cached
Low flow : 41 participants cached


## 2. Check participant comments before matching

Your `extract_mean_amplitude` function falls back to `f"sub-{i:03d}"` when
`evoked.comment` is empty. That fallback is index-based, not identity-based --
if it ever triggers, matching by comment below would silently pair the wrong
people across conditions. Check for this first:


In [6]:
from collections import Counter

for label, evoked_list in [("High Flow", evokeds_high), ("Low Flow", evokeds_low)]:
    comments = [ev.comment for ev in evoked_list]
    n_total = len(comments)
    n_unique = len(set(comments))
    n_fallback = sum(1 for c in comments if not c or c.startswith("sub-"))

    print(f"{label}: {n_total} evokeds, {n_unique} unique comments, "
          f"{n_fallback} possibly-fallback label(s)")

    if n_unique != n_total:
        dupes = [c for c, cnt in Counter(comments).items() if cnt > 1]
        print(f"  WARNING -- duplicate comment(s) found: {dupes}")

    if n_fallback > 0:
        fallback_examples = [c for c in comments if not c or c.startswith("sub-")][:5]
        print(f"  WARNING -- possible fallback labels (not real participant IDs): "
              f"{fallback_examples}")

print("\nIf any warnings printed above, resolve them before trusting the "
      "participant-matching step below -- otherwise cross-condition pairing "
      "may be silently wrong.")

High Flow: 41 evokeds, 41 unique comments, 0 possibly-fallback label(s)
Low Flow: 41 evokeds, 41 unique comments, 0 possibly-fallback label(s)

If any warnings printed above, resolve them before trusting the participant-matching step below -- otherwise cross-condition pairing may be silently wrong.


## 3. Match participants by ID

Matched explicitly by `evoked.comment`, not by assuming list order is aligned
between the High-flow and Low-flow caches.


In [7]:
high_by_subj = {ev.comment: ev for ev in evokeds_high}
low_by_subj  = {ev.comment: ev for ev in evokeds_low}

common_subjects = sorted(set(high_by_subj) & set(low_by_subj))
missing_from_high = set(low_by_subj) - set(high_by_subj)
missing_from_low  = set(high_by_subj) - set(low_by_subj)

if missing_from_high or missing_from_low:
    print(f"WARNING: {len(missing_from_high)} subject(s) present in Low but not High "
          f"({sorted(missing_from_high)}); "
          f"{len(missing_from_low)} present in High but not Low "
          f"({sorted(missing_from_low)}). "
          f"These are excluded from the paired test below.")

print(f"Paired subjects entering the test: {len(common_subjects)}")

Paired subjects entering the test: 41


## 4. Downsample, then build the per-subject flow-effect array

Shape needed: `(n_subjects, n_times, n_channels)`. Downsampled from 1024 Hz to 256 Hz first -- see code comment for rationale.

In [8]:
# Downsample: original sfreq is 1024 Hz, but the data is already low-pass
# filtered at 40 Hz, so anything above ~80 Hz Nyquist is redundant for this
# test. Resampling to 256 Hz (a clean 4x divisor) cuts compute substantially
# with a comfortable safety margin. Applied identically to every subject and
# both conditions -- a fixed preprocessing step, not tuned per contrast.
TARGET_SFREQ = 256

high_by_subj_rs = {
    s: high_by_subj[s].copy().pick("eeg", exclude="bads").resample(TARGET_SFREQ)
    for s in common_subjects
}
low_by_subj_rs = {
    s: low_by_subj[s].copy().pick("eeg", exclude="bads").resample(TARGET_SFREQ)
    for s in common_subjects
}

X = np.stack([
    (high_by_subj_rs[s].data - low_by_subj_rs[s].data).T
    for s in common_subjects
], axis=0)

print(f"Data shape (subjects, times, channels): {X.shape}")

info = high_by_subj_rs[common_subjects[0]].info
times = high_by_subj_rs[common_subjects[0]].times


Data shape (subjects, times, channels): (41, 256, 62)


## 5. Channel adjacency

In [9]:
adjacency, ch_names = find_ch_adjacency(info, ch_type="eeg")

Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.
-- number of adjacent vertices : 62


## 6. Run the test

TFCE (no cluster-forming threshold to justify), two-tailed (our MMN result
ran opposite to the a priori hypothesis, so a one-tailed test would have
missed our own effect -- see decision log).


In [ ]:
threshold = dict(start=0, step=0.2)  # TFCE
n_permutations = 5000

T_obs, clusters, cluster_p_values, H0 = spatio_temporal_cluster_1samp_test(
    X,
    threshold=threshold,
    adjacency=adjacency,
    n_permutations=n_permutations,
    tail=0,
    n_jobs=-1,
    seed=42,
    verbose=True,
)

## 7. Inspect significant clusters

The time range of a significant cluster also doubles as the R2-4 time-course argument.

In [ ]:
good_clusters = np.where(cluster_p_values < 0.05)[0]
print(f"{len(good_clusters)} significant cluster(s) out of {len(clusters)} total.")

for i in good_clusters:
    time_inds, ch_inds = clusters[i]
    cluster_times = times[np.unique(time_inds)]
    cluster_chs = [ch_names[c] for c in np.unique(ch_inds)]
    print(f"\nCluster {i}: p = {cluster_p_values[i]:.4f}")
    print(f"  Time range: {cluster_times.min()*1000:.0f} to {cluster_times.max()*1000:.0f} ms")
    print(f"  Channels: {cluster_chs}")

## 8. Reusable function: run for other contrasts / restricted ROIs

Turns the manual procedure above into a callable function, so testing
Novel-Standard or a spatially-restricted ROI doesn't mean re-copying the
whole pipeline a third time.

In [10]:
def run_cluster_test(
    diff_event: str,
    channel_picks: list[str] | None = None,
    tmin_ms: float | None = None,
    tmax_ms: float | None = None,
    target_sfreq: int = 256,
    n_permutations: int = 5000,
    seed: int = 42,
) -> dict:
    """Run the flow-effect cluster-based permutation test for a given
    difference-wave contrast (e.g. 'deviant-standard' or 'novel-standard'),
    optionally restricted to a specified a priori channel ROI and/or a
    broad, literature-justified time range (NOT a narrow window shaped by
    where this dataset's own peak happens to sit -- e.g. 0-400ms as the
    plausible range for N1/MMN/P3a-type components, regardless of what
    this data shows).

    channel_picks=None -> whole-scalp EEG (bads excluded).
    tmin_ms/tmax_ms=None -> full epoch, no temporal restriction.
    """
    t0 = time.time()
    evokeds_high_raw = evokeds_diff[COND_HIGH_LABEL][f"{cond_key_high}_{diff_event}"]
    evokeds_low_raw  = evokeds_diff[COND_LOW_LABEL][f"{cond_key_low}_{diff_event}"]

    high_by_subj = {ev.comment: ev for ev in evokeds_high_raw}
    low_by_subj  = {ev.comment: ev for ev in evokeds_low_raw}

    common_subjects = sorted(set(high_by_subj) & set(low_by_subj))
    missing_from_high = set(low_by_subj) - set(high_by_subj)
    missing_from_low  = set(high_by_subj) - set(low_by_subj)
    if missing_from_high or missing_from_low:
        print(f"[{diff_event}] WARNING: {len(missing_from_high)} missing from High, "
              f"{len(missing_from_low)} missing from Low. Excluded from test.")
    print(f"[{diff_event}] Paired subjects: {len(common_subjects)}")

    def _prep(ev):
        e = ev.copy()
        e.pick(channel_picks) if channel_picks is not None else e.pick("eeg", exclude="bads")
        e = e.resample(target_sfreq)
        if tmin_ms is not None or tmax_ms is not None:
            e.crop(
                tmin=tmin_ms / 1000 if tmin_ms is not None else None,
                tmax=tmax_ms / 1000 if tmax_ms is not None else None,
            )
        return e

    high_rs = {s: _prep(high_by_subj[s]) for s in common_subjects}
    low_rs  = {s: _prep(low_by_subj[s])  for s in common_subjects}

    X = np.stack([(high_rs[s].data - low_rs[s].data).T for s in common_subjects], axis=0)
    info = high_rs[common_subjects[0]].info
    times = high_rs[common_subjects[0]].times

    adjacency, ch_names = find_ch_adjacency(info, ch_type="eeg")
    threshold = dict(start=0, step=0.2)  # TFCE

    T_obs, clusters, cluster_p_values, H0 = spatio_temporal_cluster_1samp_test(
        X, threshold=threshold, adjacency=adjacency, n_permutations=n_permutations,
        tail=0, n_jobs=-1, seed=seed, verbose=True,
    )

    good = np.where(cluster_p_values < 0.05)[0]
    print(f"[{diff_event} | picks={channel_picks} | crop={tmin_ms}-{tmax_ms}ms] "
          f"{len(good)} significant cluster(s) out of {len(clusters)} total.")
    for i in good:
        time_inds, ch_inds = clusters[i]
        ct = times[np.unique(time_inds)]
        cc = [ch_names[c] for c in np.unique(ch_inds)]
        print(f"  Cluster {i}: p={cluster_p_values[i]:.4f}, "
              f"{ct.min()*1000:.0f}-{ct.max()*1000:.0f} ms, channels: {cc}")

    elapsed = time.time() - t0
    print(f"Done in {int(elapsed//60)}m {elapsed%60:.1f}s")

    return dict(T_obs=T_obs, clusters=clusters, cluster_p_values=cluster_p_values,
                times=times, ch_names=ch_names, common_subjects=common_subjects, X=X)

## 9. Run: Novel-Standard, whole-scalp (not yet tested)

In [ ]:
result_novel_full = run_cluster_test("novel-standard")

## 10. Plot the actual subtracted line (do this before the ROI test)

The direct quantity the cluster test evaluates: grand-average (High - Low),
averaged across your chosen electrodes -- not two overlaid traces.

**TODO: replace `FRONTAL_ROI` with your actual electrode list, and
`window_ms` with your originally chosen window.**

In [ ]:
import matplotlib.pyplot as plt
from mne import grand_average

PLOTS_DIR = find_repo_root() / "publication" / "plots" / "exploratory" / "cluster_test_diagnostics"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving diagnostic plots to: {PLOTS_DIR}")

def plot_flow_difference(diff_event: str, channel_picks: list[str], window_ms=None, save=True):
    evokeds_high_raw = evokeds_diff[COND_HIGH_LABEL][f"{cond_key_high}_{diff_event}"]
    evokeds_low_raw  = evokeds_diff[COND_LOW_LABEL][f"{cond_key_low}_{diff_event}"]

    ga_high = grand_average([ev.copy().pick(channel_picks) for ev in evokeds_high_raw])
    ga_low  = grand_average([ev.copy().pick(channel_picks) for ev in evokeds_low_raw])

    flow_diff = ga_high.copy()
    flow_diff.data = ga_high.data - ga_low.data

    mean_over_chs = flow_diff.data.mean(axis=0) * 1e6  # V -> uV
    times_ms = flow_diff.times * 1000

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(times_ms, mean_over_chs, color="black", linewidth=1.5)
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.axvline(0, color="grey", linewidth=0.8, linestyle="--")
    if window_ms is not None:
        ax.axvspan(window_ms[0], window_ms[1], color="orange", alpha=0.2, label="Window (diagnostic only)")
        ax.legend()
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("High minus Low flow amplitude (uV)")
    ax.set_title(f"Flow effect on {diff_event}, averaged across {channel_picks}")
    fig.tight_layout()

    if save:
        safe_name = diff_event.replace(" ", "_")
        fpath = PLOTS_DIR / f"flow_diff_{safe_name}.png"
        fig.savefig(fpath, dpi=150)
        print(f"Saved: {fpath}")

    plt.show()
    return flow_diff

# ROI per contrast -- distinct topography: MMN (Deviant-Standard) fronto-central,
# novelty response (Novel-Standard) more central. Confirm both are grounded in
# prior literature/latency expectations, not tuned to this dataset's own peaks.
FRONTAL_ROI = ["Fz", "F1", "F2", "FCz", "FC1", "FC2"]   # Deviant-Standard (MMN)
CENTRAL_ROI = ["FCz", "FC1", "FC2", "Cz", "C1", "C2"]   # Novel-Standard

WINDOW_MS_DEV   = (100, 200)  # TODO: confirm against literature/latency, not just "looks right here"
WINDOW_MS_NOVEL = (100, 200)  # TODO: same

flow_diff_dev   = plot_flow_difference(f"{DEVIANT_LABEL}-standard", FRONTAL_ROI, WINDOW_MS_DEV)
flow_diff_novel = plot_flow_difference("novel-standard", CENTRAL_ROI, WINDOW_MS_NOVEL)

## 10b. Quantify: peak-to-baseline ratio

Peak amplitude within the chosen window, divided by the standard deviation
of the same line during a period where nothing should be happening (the
pre-stimulus baseline). A ratio clearly above ~1 (roughly 2-3x) is
consistent with a real signal standing out from the noise floor; a ratio
near 1 means what looked like a bump by eye is about the same size as
ordinary baseline wobble.

**TODO: confirm `BASELINE_MS` matches your actual pre-stimulus baseline
window (placeholder: -200 to 0 ms).**

In [ ]:
BASELINE_MS = (-200, 0)  # TODO: confirm this matches your actual baseline window

def peak_to_baseline_ratio(flow_diff, window_ms, baseline_ms):
    """Peak |amplitude| in window_ms, divided by SD of the same line during
    baseline_ms. Both computed on the same (already electrode-averaged)
    difference line."""
    data_uv = flow_diff.data.mean(axis=0) * 1e6  # uV, averaged across the picked channels
    times_ms = flow_diff.times * 1000

    win_mask = (times_ms >= window_ms[0]) & (times_ms <= window_ms[1])
    base_mask = (times_ms >= baseline_ms[0]) & (times_ms <= baseline_ms[1])

    peak_amp = np.max(np.abs(data_uv[win_mask]))
    baseline_sd = np.std(data_uv[base_mask])
    ratio = peak_amp / baseline_sd

    print(f"  Peak amplitude in window : {peak_amp:.3f} uV")
    print(f"  Baseline SD              : {baseline_sd:.3f} uV")
    print(f"  Peak-to-baseline ratio   : {ratio:.2f}")
    return ratio

print("Deviant-Standard:")
ratio_dev = peak_to_baseline_ratio(flow_diff_dev, WINDOW_MS_DEV, BASELINE_MS)

print("\nNovel-Standard:")
ratio_novel = peak_to_baseline_ratio(flow_diff_novel, WINDOW_MS_NOVEL, BASELINE_MS)

## 11. If the plot confirms it: spatially-restricted cluster test

Same method, same TFCE, same two-tailed test -- just restricted to the a
priori ROI instead of all 62 channels. This reduces the multiple-comparisons
burden without choosing anything based on this dataset's own results.

In [ ]:
result_dev_roi   = run_cluster_test(f"{DEVIANT_LABEL}-standard", channel_picks=FRONTAL_ROI)
result_novel_roi = run_cluster_test("novel-standard", channel_picks=CENTRAL_ROI)

## 12. Re-run with a broad, literature-justified time crop

0-400ms post-stimulus as the plausible range for N1/MMN/P3a-type components
-- chosen because no relevant component of interest occurs later than this,
regardless of what this dataset shows, not because it brackets this
dataset's own peaks. Removes correction burden wasted on baseline and
late-epoch stretches where nothing relevant could plausibly occur.

In [12]:
FRONTAL_ROI = ["Fz", "F1", "F2", "FCz", "FC1", "FC2"]   # Deviant-Standard (MMN)
CENTRAL_ROI = ["FCz", "FC1", "FC2", "Cz", "C1", "C2"]   # Novel-Standard

result_dev_roi_crop = run_cluster_test(
    f"{DEVIANT_LABEL}-standard", channel_picks=FRONTAL_ROI, tmin_ms=0, tmax_ms=400
)
result_novel_roi_crop = run_cluster_test(
    "novel-standard", channel_picks=CENTRAL_ROI, tmin_ms=0, tmax_ms=400
)

[deviant-standard] Paired subjects: 41
Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.
-- number of adjacent vertices : 6
stat_fun(H1): min=-3.1574629632497295 max=0.3425985262803995
Running initial clustering …
Using 16 thresholds from 0.00 to 3.00 for TFCE computation (h_power=2.00, e_power=0.50)
Found 618 clusters


C:\Users\anguyen\pyvenv\mscheltienne\eeg-flow\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|████████████████████████████████████████████████████████████████| Permuting : 4999/4999 [01:24<00:00,   59.20it/s]


[deviant-standard | picks=['Fz', 'F1', 'F2', 'FCz', 'FC1', 'FC2'] | crop=0-400ms] 84 significant cluster(s) out of 618 total.
  Cluster 119: p=0.0478, 73-73 ms, channels: ['FC2']
  Cluster 125: p=0.0454, 77-77 ms, channels: ['FC2']
  Cluster 131: p=0.0454, 81-81 ms, channels: ['FC2']
  Cluster 198: p=0.0292, 128-128 ms, channels: ['Fz']
  Cluster 199: p=0.0374, 128-128 ms, channels: ['F1']
  Cluster 200: p=0.0374, 128-128 ms, channels: ['F2']
  Cluster 201: p=0.0292, 128-128 ms, channels: ['FCz']
  Cluster 202: p=0.0374, 128-128 ms, channels: ['FC1']
  Cluster 203: p=0.0374, 128-128 ms, channels: ['FC2']
  Cluster 204: p=0.0292, 132-132 ms, channels: ['Fz']
  Cluster 205: p=0.0292, 132-132 ms, channels: ['F1']
  Cluster 206: p=0.0374, 132-132 ms, channels: ['F2']
  Cluster 207: p=0.0292, 132-132 ms, channels: ['FCz']
  Cluster 208: p=0.0292, 132-132 ms, channels: ['FC1']
  Cluster 209: p=0.0374, 132-132 ms, channels: ['FC2']
  Cluster 210: p=0.0292, 136-136 ms, channels: ['Fz']
  Clust

100%|████████████████████████████████████████████████████████████████| Permuting : 4999/4999 [01:34<00:00,   52.91it/s]


[novel-standard | picks=['FCz', 'FC1', 'FC2', 'Cz', 'C1', 'C2'] | crop=0-400ms] 0 significant cluster(s) out of 618 total.
Done in 1m 35.1s


In [25]:
def summarize_significant_range(result: dict, label: str):
    good = np.where(result["cluster_p_values"] < 0.05)[0]
    all_time_inds, all_ch_inds = [], []
    for i in good:
        time_inds, ch_inds = result["clusters"][i]
        all_time_inds.extend(np.atleast_1d(time_inds))
        all_ch_inds.extend(np.atleast_1d(ch_inds))

    times = result["times"]
    ch_names = result["ch_names"]
    sig_times_ms = times[sorted(set(all_time_inds))] * 1000
    sig_channels = sorted(set(ch_names[c] for c in set(all_ch_inds)))

    print(f"[{label}] Significant range: {sig_times_ms.min():.0f} to {sig_times_ms.max():.0f} ms")
    print(f"[{label}] Significant channels: {sig_channels}")
    print(f"[{label}] Distinct significant timepoints: {len(set(all_time_inds))} / {len(times)}")


In [23]:
summarize_significant_range(result_dev_roi_crop, "Deviant-Standard (tight crop)")

[Deviant-Standard (tight crop)] Significant range: 73 to 194 ms
[Deviant-Standard (tight crop)] Significant channels: ['F1', 'F2', 'FC1', 'FC2', 'FCz', 'Fz']
[Deviant-Standard (tight crop)] Distinct significant timepoints: 20 / 103


In [29]:
# Novel-Standard: use the ALREADY pre-specified window (same as multiverse), not the ad hoc 0-200 crop
result_novel_roi_crop = run_cluster_test(
    "novel-standard", channel_picks=CENTRAL_ROI, tmin_ms=0, tmax_ms=200
)

[novel-standard] Paired subjects: 41
Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.
-- number of adjacent vertices : 6
stat_fun(H1): min=-3.133525181794478 max=0.3498616955524337
Running initial clustering …
Using 16 thresholds from 0.00 to 3.00 for TFCE computation (h_power=2.00, e_power=0.50)
Found 312 clusters


100%|████████████████████████████████████████████████████████████████| Permuting : 4999/4999 [00:44<00:00,  111.69it/s]


[novel-standard | picks=['FCz', 'FC1', 'FC2', 'Cz', 'C1', 'C2'] | crop=0-200ms] 26 significant cluster(s) out of 312 total.
  Cluster 74: p=0.0494, 46-46 ms, channels: ['FC2']
  Cluster 75: p=0.0494, 46-46 ms, channels: ['Cz']
  Cluster 77: p=0.0494, 46-46 ms, channels: ['C2']
  Cluster 78: p=0.0494, 50-50 ms, channels: ['FCz']
  Cluster 79: p=0.0494, 50-50 ms, channels: ['FC1']
  Cluster 80: p=0.0472, 50-50 ms, channels: ['FC2']
  Cluster 81: p=0.0472, 50-50 ms, channels: ['Cz']
  Cluster 82: p=0.0494, 50-50 ms, channels: ['C1']
  Cluster 83: p=0.0494, 50-50 ms, channels: ['C2']
  Cluster 86: p=0.0494, 54-54 ms, channels: ['FC2']
  Cluster 87: p=0.0494, 54-54 ms, channels: ['Cz']
  Cluster 88: p=0.0494, 54-54 ms, channels: ['C1']
  Cluster 89: p=0.0494, 54-54 ms, channels: ['C2']
  Cluster 104: p=0.0494, 65-65 ms, channels: ['FC2']
  Cluster 105: p=0.0494, 65-65 ms, channels: ['Cz']
  Cluster 108: p=0.0494, 69-69 ms, channels: ['FCz']
  Cluster 110: p=0.0448, 69-69 ms, channels: ['FC2

In [30]:
summarize_significant_range(result_novel_roi_crop, "Novel-Standard (tight crop)")

[Novel-Standard (tight crop)] Significant range: 46 to 77 ms
[Novel-Standard (tight crop)] Significant channels: ['C1', 'C2', 'Cz', 'FC1', 'FC2', 'FCz']
[Novel-Standard (tight crop)] Distinct significant timepoints: 7 / 52


In [26]:
# Novel-Standard: use the ALREADY pre-specified window (same as multiverse), not the ad hoc 0-200 crop
result_novel_roi_crop_correct = run_cluster_test(
    "novel-standard", channel_picks=CENTRAL_ROI, tmin_ms=120, tmax_ms=180
)

[novel-standard] Paired subjects: 41
Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.
-- number of adjacent vertices : 6
stat_fun(H1): min=-2.8807691772622706 max=-0.8542469574203012
Running initial clustering …
Using 15 thresholds from 0.00 to 2.80 for TFCE computation (h_power=2.00, e_power=0.50)
Found 96 clusters


100%|████████████████████████████████████████████████████████████████| Permuting : 4999/4999 [00:13<00:00,  365.54it/s]

[novel-standard | picks=['FCz', 'FC1', 'FC2', 'Cz', 'C1', 'C2'] | crop=120-180ms] 40 significant cluster(s) out of 96 total.
  Cluster 50: p=0.0414, 151-151 ms, channels: ['FC2']
  Cluster 53: p=0.0414, 151-151 ms, channels: ['C2']
  Cluster 54: p=0.0414, 155-155 ms, channels: ['FCz']
  Cluster 55: p=0.0414, 155-155 ms, channels: ['FC1']
  Cluster 56: p=0.0298, 155-155 ms, channels: ['FC2']
  Cluster 57: p=0.0298, 155-155 ms, channels: ['Cz']
  Cluster 58: p=0.0414, 155-155 ms, channels: ['C1']
  Cluster 59: p=0.0298, 155-155 ms, channels: ['C2']
  Cluster 60: p=0.0298, 159-159 ms, channels: ['FCz']
  Cluster 61: p=0.0298, 159-159 ms, channels: ['FC1']
  Cluster 62: p=0.0258, 159-159 ms, channels: ['FC2']
  Cluster 63: p=0.0258, 159-159 ms, channels: ['Cz']
  Cluster 64: p=0.0258, 159-159 ms, channels: ['C1']
  Cluster 65: p=0.0258, 159-159 ms, channels: ['C2']
  Cluster 66: p=0.0258, 163-163 ms, channels: ['FCz']
  Cluster 67: p=0.0258, 163-163 ms, channels: ['FC1']
  Cluster 68: p=0.

In [27]:
summarize_significant_range(result_novel_roi_crop_correct, "Novel-Standard (tight crop)")

[Novel-Standard (tight crop)] Significant range: 151 to 179 ms
[Novel-Standard (tight crop)] Significant channels: ['C1', 'C2', 'Cz', 'FC1', 'FC2', 'FCz']
[Novel-Standard (tight crop)] Distinct significant timepoints: 8 / 16


In [28]:
# Whole-scalp checks: reuse the SAME crop bounds as whichever ROI test succeeded --
# NOT the discovered significant range (73-194ms), which would itself be circular
result_dev_full_crop = run_cluster_test(
    f"{DEVIANT_LABEL}-standard", channel_picks=None, tmin_ms=0, tmax_ms=400
)
result_novel_full_crop = run_cluster_test(
    "novel-standard", channel_picks=None, tmin_ms=120, tmax_ms=180
)

[deviant-standard] Paired subjects: 41
Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.
-- number of adjacent vertices : 62
stat_fun(H1): min=-3.859006851281919 max=2.214846030052654
Running initial clustering …
Using 20 thresholds from 0.00 to 3.80 for TFCE computation (h_power=2.00, e_power=0.50)
Found 6386 clusters


100%|████████████████████████████████████████████████████████████████| Permuting : 4999/4999 [10:23<00:00,    8.02it/s]


[deviant-standard | picks=None | crop=0-400ms] 756 significant cluster(s) out of 6386 total.
  Cluster 1184: p=0.0440, 73-73 ms, channels: ['F4']
  Cluster 1188: p=0.0416, 73-73 ms, channels: ['FC2']
  Cluster 1189: p=0.0440, 73-73 ms, channels: ['FC6']
  Cluster 1191: p=0.0480, 73-73 ms, channels: ['C3']
  Cluster 1192: p=0.0480, 73-73 ms, channels: ['Cz']
  Cluster 1193: p=0.0392, 73-73 ms, channels: ['C4']
  Cluster 1196: p=0.0440, 73-73 ms, channels: ['CP1']
  Cluster 1197: p=0.0416, 73-73 ms, channels: ['CP2']
  Cluster 1198: p=0.0440, 73-73 ms, channels: ['CP6']
  Cluster 1200: p=0.0480, 73-73 ms, channels: ['P3']
  Cluster 1201: p=0.0480, 73-73 ms, channels: ['Pz']
  Cluster 1202: p=0.0440, 73-73 ms, channels: ['P4']
  Cluster 1212: p=0.0480, 73-73 ms, channels: ['F1']
  Cluster 1214: p=0.0440, 73-73 ms, channels: ['F6']
  Cluster 1216: p=0.0480, 73-73 ms, channels: ['FCz']
  Cluster 1217: p=0.0392, 73-73 ms, channels: ['FC4']
  Cluster 1219: p=0.0480, 73-73 ms, channels: ['C1']

100%|████████████████████████████████████████████████████████████████| Permuting : 4999/4999 [01:41<00:00,   49.16it/s]

[novel-standard | picks=None | crop=120-180ms] 388 significant cluster(s) out of 992 total.
  Cluster 32: p=0.0440, 120-120 ms, channels: ['AF8']
  Cluster 55: p=0.0304, 120-120 ms, channels: ['FT8']
  Cluster 78: p=0.0238, 124-124 ms, channels: ['T8']
  Cluster 82: p=0.0440, 124-124 ms, channels: ['CP6']
  Cluster 90: p=0.0304, 124-124 ms, channels: ['O2']
  Cluster 94: p=0.0220, 124-124 ms, channels: ['AF8']
  Cluster 105: p=0.0440, 124-124 ms, channels: ['C6']
  Cluster 114: p=0.0440, 124-124 ms, channels: ['PO4']
  Cluster 115: p=0.0304, 124-124 ms, channels: ['PO6']
  Cluster 117: p=0.0238, 124-124 ms, channels: ['FT8']
  Cluster 119: p=0.0440, 124-124 ms, channels: ['TP8']
  Cluster 121: p=0.0440, 124-124 ms, channels: ['PO8']
  Cluster 131: p=0.0440, 128-128 ms, channels: ['F8']
  Cluster 135: p=0.0238, 128-128 ms, channels: ['FC6']
  Cluster 139: p=0.0238, 128-128 ms, channels: ['C4']
  Cluster 140: p=0.0174, 128-128 ms, channels: ['T8']
  Cluster 143: p=0.0440, 128-128 ms, cha

In [32]:
summarize_significant_range(result_dev_full_crop, "Deviant-Standard (full crop)")
print()
summarize_significant_range(result_novel_full_crop, "Novel-Standard (full crop)")

[Deviant-Standard (full crop)] Significant range: 73 to 198 ms
[Deviant-Standard (full crop)] Significant channels: ['AF3', 'AF4', 'AF7', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'CP1', 'CP2', 'CP3', 'CP4', 'CP5', 'CP6', 'CPz', 'Cz', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'FC1', 'FC2', 'FC3', 'FC4', 'FC5', 'FC6', 'FCz', 'FT7', 'FT8', 'Fp1', 'Fp2', 'Fpz', 'Fz', 'O1', 'O2', 'Oz', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P8', 'PO3', 'PO4', 'PO5', 'PO6', 'POz', 'Pz', 'T7', 'T8', 'TP8']
[Deviant-Standard (full crop)] Distinct significant timepoints: 33 / 103

[Novel-Standard (full crop)] Significant range: 120 to 179 ms
[Novel-Standard (full crop)] Significant channels: ['AF4', 'AF7', 'AF8', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'CP1', 'CP2', 'CP3', 'CP4', 'CP5', 'CP6', 'CPz', 'Cz', 'F1', 'F2', 'F3', 'F4', 'F6', 'F8', 'FC1', 'FC2', 'FC3', 'FC4', 'FC5', 'FC6', 'FCz', 'FT8', 'Fp2', 'Fpz', 'Fz', 'O1', 'O2', 'Oz', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P8', 'PO3', 'PO4', 'PO6', 'PO8', 'POz', 'Pz', 

In [33]:
from scipy import stats

def run_cluster_test_fixed_threshold(
    diff_event: str,
    channel_picks: list[str],
    tmin_ms: float,
    tmax_ms: float,
    alpha_cluster_forming: float = 0.05,
    target_sfreq: int = 256,
    n_permutations: int = 5000,
    seed: int = 42,
) -> dict:
    """Primary confirmatory test: fixed-threshold cluster-based permutation
    (Maris & Oostenveld, 2007), scoped to a genuine a priori component
    window -- not TFCE, not a wide invented crop. Reports real cluster mass.
    """
    t0 = time.time()
    
    evokeds_high_raw = evokeds_diff[COND_HIGH_LABEL][f"{cond_key_high}_{diff_event}"]
    evokeds_low_raw  = evokeds_diff[COND_LOW_LABEL][f"{cond_key_low}_{diff_event}"]

    high_by_subj = {ev.comment: ev for ev in evokeds_high_raw}
    low_by_subj  = {ev.comment: ev for ev in evokeds_low_raw}
    common_subjects = sorted(set(high_by_subj) & set(low_by_subj))
    n_subj = len(common_subjects)

    def _prep(ev):
        e = ev.copy().pick(channel_picks).resample(target_sfreq)
        e.crop(tmin=tmin_ms / 1000, tmax=tmax_ms / 1000)
        return e

    high_rs = {s: _prep(high_by_subj[s]) for s in common_subjects}
    low_rs  = {s: _prep(low_by_subj[s])  for s in common_subjects}

    X = np.stack([(high_rs[s].data - low_rs[s].data).T for s in common_subjects], axis=0)
    info = high_rs[common_subjects[0]].info
    times = high_rs[common_subjects[0]].times

    adjacency, ch_names = find_ch_adjacency(info, ch_type="eeg")

    # Fixed cluster-forming threshold: the t-value for the given alpha and df,
    # not TFCE's continuous integration. This is the standard, familiar choice.
    threshold = stats.t.ppf(1 - alpha_cluster_forming / 2, df=n_subj - 1)

    T_obs, clusters, cluster_p_values, H0 = spatio_temporal_cluster_1samp_test(
        X, threshold=threshold, adjacency=adjacency, n_permutations=n_permutations,
        tail=0, n_jobs=-1, seed=seed, verbose=True,
    )

    good = np.where(cluster_p_values < 0.05)[0]
    print(f"[{diff_event} | {tmin_ms}-{tmax_ms}ms | fixed threshold t={threshold:.3f}] "
          f"{len(good)} significant cluster(s) out of {len(clusters)} total.")
    for i in good:
        time_inds, ch_inds = clusters[i]
        ct = times[np.unique(time_inds)]
        cc = [ch_names[c] for c in np.unique(ch_inds)]
        cluster_mass = T_obs[time_inds, ch_inds].sum()  # genuine cluster mass, real clusters now
        print(f"  Cluster {i}: p={cluster_p_values[i]:.4f}, cluster mass={cluster_mass:.2f}, "
              f"{ct.min()*1000:.0f}-{ct.max()*1000:.0f} ms, channels: {cc}")

    elapsed = time.time() - t0
    print(f"Done in {int(elapsed//60)}m {elapsed%60:.1f}s")

    return dict(T_obs=T_obs, clusters=clusters, cluster_p_values=cluster_p_values,
                times=times, ch_names=ch_names, common_subjects=common_subjects, X=X)


# --- Primary test, real a priori windows, one per component ---
result_dev_N1  = run_cluster_test_fixed_threshold(f"{DEVIANT_LABEL}-standard", FRONTAL_ROI, 90, 130)
result_dev_MMN = run_cluster_test_fixed_threshold(f"{DEVIANT_LABEL}-standard", FRONTAL_ROI, 130, 190)
result_dev_P3a = run_cluster_test_fixed_threshold(f"{DEVIANT_LABEL}-standard", FRONTAL_ROI, 200, 280)

result_nov_N1MMN = run_cluster_test_fixed_threshold("novel-standard", CENTRAL_ROI, 120, 180)
result_nov_P3a   = run_cluster_test_fixed_threshold("novel-standard", CENTRAL_ROI, 180, 280)

Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.
-- number of adjacent vertices : 6
stat_fun(H1): min=-2.834857450301468 max=-1.564137335686019
Running initial clustering …
Found 2 clusters


100%|████████████████████████████████████████████████████████████████| Permuting : 4999/4999 [00:09<00:00,  528.42it/s]


[deviant-standard | 90-130ms | fixed threshold t=2.021] 1 significant cluster(s) out of 2 total.
  Cluster 0: p=0.0304, cluster mass=-78.41, 89-112 ms, channels: ['Fz', 'F1', 'F2', 'FCz', 'FC1', 'FC2']
Done in 0m 10.0s
Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.
-- number of adjacent vertices : 6
stat_fun(H1): min=-3.027126804575769 max=-2.0869716091624317
Running initial clustering …
Found 1 cluster


100%|███████████████████████████████████████████████████████████████| Permuting : 4999/4999 [00:00<00:00, 24729.50it/s]


[deviant-standard | 130-190ms | fixed threshold t=2.021] 1 significant cluster(s) out of 1 total.
  Cluster 0: p=0.0028, cluster mass=-262.66, 128-190 ms, channels: ['Fz', 'F1', 'F2', 'FCz', 'FC1', 'FC2']
Done in 0m 0.6s
Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.
-- number of adjacent vertices : 6
stat_fun(H1): min=-2.213669833144496 max=-0.3128993599931003
Running initial clustering …
Found 3 clusters


...

[deviant-standard | 200-280ms | fixed threshold t=2.021] 0 significant cluster(s) out of 3 total.
Done in 0m 0.9s
Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.
-- number of adjacent vertices : 6
stat_fun(H1): min=-2.8807691772622706 max=-0.8542469574203012
Running initial clustering …
Found 1 cluster


100%|███████████████████████████████████████████████████████████████| Permuting : 4999/4999 [00:00<00:00, 21950.03it/s]


[novel-standard | 120-180ms | fixed threshold t=2.021] 1 significant cluster(s) out of 1 total.
  Cluster 0: p=0.0200, cluster mass=-136.57, 128-179 ms, channels: ['FCz', 'FC1', 'FC2', 'Cz', 'C1', 'C2']
Done in 0m 0.7s
Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.
-- number of adjacent vertices : 6
stat_fun(H1): min=-2.3446428075534396 max=-0.158710372978733
Running initial clustering …
Found 1 cluster


100%|███████████████████████████████████████████████████████████████| Permuting : 4999/4999 [00:00<00:00, 14505.40it/s]

[novel-standard | 180-280ms | fixed threshold t=2.021] 0 significant cluster(s) out of 1 total.
Done in 0m 0.7s
